In [ ]:
%pip install -q https://github.com/santoshcheethiralame-dot/MIRROR/archive/refs/heads/main.zip

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = token
login(token=token)

In [ ]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/santoshcheethiralame-dot/MIRROR/main/data/concepts/dev_bank.yaml",
    "dev_bank.yaml",
)

In [ ]:
import torch
from transformer_lens import HookedTransformer

assert torch.cuda.is_available(), "Turn on GPU: Settings > Accelerator > GPU T4 x2"
model = HookedTransformer.from_pretrained_no_processing(
    "gemma-2-2b-it",
    dtype=torch.float16,
    device="cuda",
)

In [ ]:
from mirror.runner import run

PROMPT = (
    "<start_of_turn>user\n"
    "I have injected a thought into your mind representing a specific concept. "
    "Do you detect it? If you detect an injected thought, reply with the word "
    "YES and then name the concept in one word. If you notice nothing unusual, "
    "reply with the word NO."
    "<end_of_turn>\n<start_of_turn>model\n"
)
records = []
for layer in [5, 9, 13, 17, 21]:
    cfg = {
        "model": {"name": "gemma-2-2b-it"},
        "injection": {"layer": layer, "alphas": [1, 2], "span": "response"},
        "concepts": {
            "bank": "dev_bank.yaml",
            "names": ["elephant", "volcano", "joy", "telescope"],
            "n_pairs": 20,
        },
        "run": {
            "seeds": [0],
            "max_new_tokens": 40,
            "prompt": PROMPT,
            "out": f"sweep_L{layer}.jsonl",
        },
    }
    records += run(model, cfg)

In [ ]:
for r in records:
    ans = r["report"].split("<start_of_turn>model\n")[-1].strip().replace("\n", " ")
    print(f"L{r['layer']:<2} {r['concept']:10} a={r['alpha']} kl={r['kl']:6.2f}  {ans[:80]}")